# 02. Pipeline CPU -> CUDA: secuencial contra acelerado

Proyecto 3 CC3069, entregable de código. Este notebook ejecuta de principio a fin el
experimento del proyecto sobre **una sola máquina**, para que todos los tiempos sean comparables.

**Regla de diseño (40 % de la nota):** los archivos se preparan en la **CPU** (varios núcleos) y la
**GPU** se usa solo para la matemática de la red (forward, backward, Adam).

| Fase | Dónde corre | Código |
|---|---|---|
| A. Preparación de datos (leer JPEG, decodificar, resize, normalizar, armar tensores) | CPU, N workers | `src/asl/torch_data.py` |
| B. Configuración CUDA y entrenamiento (transferencia H2D, forward, backward, Adam) | GPU (CUDA + cuDNN) | `scripts/benchmark.py`, `src/asl/torch_models.py` |
| Línea base secuencial original | CPU, Keras/TensorFlow | `notebooks/01_baseline_keras.ipynb` |
| CNN desde cero (explícita con CUDA): NumPy contra CuPy | CPU / GPU | `src/scratch/`, notebook `03_cnn_scratch.ipynb` |

Secciones:

0. Configuración y verificación del hardware
1. **Fase A**: preparación de datos en CPU (1, 4, 8 y N workers)
2. Solución secuencial original (Keras en CPU)
3. **Fase B**: PyTorch en CPU (control) y en CUDA (solución acelerada)
4. CNN desde cero: NumPy (CPU) y CuPy (GPU), ver notebook 03
5. Evidencia de uso de hardware (CPU y GPU)
6. Entrenamiento completo con Early Stopping
7. Tablas de tiempos y speedup por fase

**Cómo correrlo:** ejecutar las celdas en orden. Las banderas de la sección 0 permiten saltarse
partes largas. Los resultados se guardan en `results/` con los mismos nombres que usa
`scripts/aggregate.py`, así que **sobrescriben** los de otras máquinas (es lo que se quiere).
Antes de correr: cerrar otros programas, equipo conectado a la corriente y en modo de alto rendimiento.

## 0. Configuración y verificación del hardware

In [1]:
# NOTA: corrida de evidencia reducida (ver banderas). Los tiempos del informe salen de results/*.json
import os, sys, json, time, glob, shutil, subprocess, statistics as st, inspect, tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Raíz del repo (el notebook vive en notebooks/)
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
RESULTS = ROOT / "results"
EVIDENCE = RESULTS / "evidence"
EVIDENCE.mkdir(parents=True, exist_ok=True)

# ---------------- Parámetros del experimento (igual que el resto del proyecto) ----------------
RUNS = 3                    # corridas independientes por configuración
EPOCHS = 5                  # épocas fijas, sin early stopping; se descarta la época 1 (calentamiento)
BATCH = 64
WORKERS = os.cpu_count()    # procesos para decodificar JPEG
WORKER_SWEEP = sorted({1, 4, 8, WORKERS})

# ---------------- Banderas: poner False para saltarse una sección ----------------
RUN_KERAS_CPU = False        # sección 2 (necesita TensorFlow + jupyter nbconvert en PYTHON_KERAS)
PYTHON_KERAS = sys.executable   # intérprete que tenga TensorFlow, si es otro venv cambiar aquí
RUN_TORCH_CPU = False        # sección 3, control PyTorch en CPU
RUN_TORCH_CUDA = True       # sección 3, solución acelerada
RUN_EVIDENCE = True         # sección 5, genera usage.csv y usage.png (~1 a 2 min)
RUN_FULL_TRAINING = False    # sección 6, tres CNN con early stopping

def sh(cmd, env=None, cwd=ROOT):
    # Ejecuta un comando y muestra su salida en la celda, línea a línea
    e = {**os.environ, **(env or {})}
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=e, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Falló: {' '.join(map(str, cmd))} (código {p.returncode})")

print(f"Raíz del proyecto: {ROOT}")
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | CUDA runtime {torch.version.cuda}")
print(f"Hilos lógicos de CPU: {WORKERS}")

Raíz del proyecto: /home/Tono/Downloads/asl-cuda
Python 3.13.9 | torch 2.9.1+cu128 | CUDA runtime 12.8
Hilos lógicos de CPU: 20


In [2]:
# Verificación de que el entorno reconoce el hardware. Se guarda como evidencia.
assert torch.cuda.is_available(), "PyTorch no ve la GPU: revisar el driver y la versión de torch+cu"

props = torch.cuda.get_device_properties(0)
GPU_NAME = props.name
print(f"GPU: {GPU_NAME} | VRAM {props.total_memory / 2**30:.1f} GiB | "
      f"capacidad de cómputo {props.major}.{props.minor} | multiprocesadores {props.multi_processor_count}")
print(f"cuDNN {torch.backends.cudnn.version()}")

try:
    import cupy as cp
    print(f"CuPy {cp.__version__}")
except Exception as e:
    cp = None
    print("CuPy no disponible:", e)

info = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
detalle = subprocess.run(["nvidia-smi", "-q"], capture_output=True, text=True).stdout
(EVIDENCE / "nvidia_smi_info.txt").write_text(info + "\n\n" + detalle, encoding="utf-8")
print(info)

GPU: NVIDIA GeForce RTX 4070 SUPER | VRAM 11.6 GiB | capacidad de cómputo 8.9 | multiprocesadores 56
cuDNN 91002
CuPy 14.2.0


Wed Nov 18 10:53:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...    Off |   00000000:01:00.0  On |                  N/A |
|  0%   39C    P8              7W /  220W |     890MiB /  12282MiB |     10%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Fase A: preparación de datos (CPU)

Todo lo que toca archivos ocurre aquí y **no usa la GPU**: leer el JPEG del disco, decodificarlo
(descompresión Huffman, secuencial por naturaleza), redimensionar a 64x64, normalizar a [0, 1]
y dejar el split como un tensor `float32 (N, 3, 64, 64)` en RAM. `DataLoader(num_workers=N)`
reparte las imágenes entre N procesos, un núcleo por imagen.

Se hace una pasada de calentamiento para que la caché de archivos del sistema operativo no
penalice la primera medición, y luego se mide con 1, 4, 8 y N workers.

In [3]:
from asl.data import load_splits, RANDOM_SEED
from asl.torch_data import decode_split

print(inspect.getsource(decode_split))

df_train, df_val, df_test = load_splits()

decode_split(df_train, WORKERS)          # calentamiento (no se cuenta)

filas = []
for w in WORKER_SWEEP:
    _, _, t = decode_split(df_train, w)
    filas.append({"workers": w, "segundos": round(t, 3), "img_por_s": round(len(df_train) / t)})
sweep = pd.DataFrame(filas)
sweep["speedup_vs_1"] = (sweep.segundos.iloc[0] / sweep.segundos).round(2)
display(sweep)
(RESULTS / "data_prep_workers.json").write_text(sweep.to_json(orient="records", indent=2))

# Tensores que usará el entrenamiento: train y test, decodificados con todos los workers
X_tr, y_tr, t_tr = decode_split(df_train, WORKERS)
X_te, y_te, t_te = decode_split(df_test, WORKERS)
DATA_PREP = {
    "train_decode_1_workers_s": float(sweep.loc[sweep.workers == 1, "segundos"].iloc[0]),
    f"train_decode_{WORKERS}_workers_s": float(sweep.loc[sweep.workers == WORKERS, "segundos"].iloc[0]),
    "data_prep_seconds": round(t_tr + t_te, 3),
}
print(f"Train {tuple(X_tr.shape)}, test {tuple(X_te.shape)}, dispositivo de los tensores: {X_tr.device}")
print(DATA_PREP)

def decode_split(df: pd.DataFrame, workers: int) -> Tuple[torch.Tensor, torch.Tensor, float]:
    """
    Lee, decodifica, redimensiona y normaliza un split.
    Devuelve (X float32 NCHW en [0,1], y int64, segundos).
    """
    t0 = time.perf_counter()
    loader = DataLoader(_JpegDataset(df["path"].values), batch_size=256, shuffle=False,
                        num_workers=workers, persistent_workers=False)
    X = torch.cat([b for b in loader])                            # (N, H, W, 3) uint8
    X = X.permute(0, 3, 1, 2).float().div_(255.0).contiguous()    # (N, 3, H, W)
    y = torch.from_numpy(df["label_id"].values.astype(np.int64))
    return X, y, time.perf_counter() - t0

[data] Train 11,600 | Val 1,450 | Test 1,450


,workers,segundos,img_por_s,speedup_vs_1
0,1,3.320,3494,1.00
1,4,0.956,12128,3.47
2,8,0.734,15809,4.52
3,20,0.525,22076,6.32


Train (11600, 3, 64, 64), test (1450, 3, 64, 64), dispositivo de los tensores: cpu
{'train_decode_1_workers_s': 3.32, 'train_decode_20_workers_s': 0.525, 'data_prep_seconds': 0.779}


## 2. Solución secuencial original (Keras, CPU)

Es el código del Lab 3 sin cambios (`01_baseline_keras.ipynb`), forzado a CPU con
`CUDA_VISIBLE_DEVICES=-1`. Se ejecuta en un proceso aparte porque esa variable debe fijarse antes de
importar TensorFlow y dejaría a PyTorch sin GPU en este kernel. Protocolo: 5 épocas fijas, sin
early stopping y sin pasada de validación. Deja `results/keras_cpu_run<N>.json`.

In [4]:
if RUN_KERAS_CPU:
    nb = ROOT / "notebooks" / "01_baseline_keras.ipynb"
    tmp = tempfile.mkdtemp()
    for run in range(1, RUNS + 1):
        print(f"\n######## Keras CPU, corrida {run}/{RUNS} ########")
        sh([PYTHON_KERAS, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", nb,
            "--output-dir", tmp, "--output", f"keras_run{run}.ipynb",
            "--ExecutePreprocessor.timeout=-1"],
           env={"ASL_EARLY_STOP": "0", "ASL_EPOCHS": str(EPOCHS), "ASL_RUN": str(run),
                "ASL_VALIDATE": "0"}, cwd=ROOT / "notebooks")
    shutil.rmtree(tmp, ignore_errors=True)
else:
    print("RUN_KERAS_CPU = False, se usan los JSON que ya existan en results/")

RUN_KERAS_CPU = False, se usan los JSON que ya existan en results/


## 3. Fase B: configuración CUDA y entrenamiento (PyTorch)

Mismo modelo, mismos datos y mismo código en CPU y en GPU, solo cambia `device`. Dentro de cada
época se cronometra por separado:

- `gather`: armar el lote en CPU (indexar el tensor en RAM).
- `h2d`: copia CPU -> GPU del lote (con `synchronize()` antes y después para aislarla).
- `step`: forward + backward + Adam. En GPU se mide con eventos CUDA, el tiempo real de los kernels.

Configuración CUDA: `torch.backends.cudnn.benchmark = True` (cuDNN elige el mejor algoritmo de
convolución para cada forma; por eso la época 1 es más lenta y se descarta), el modelo se mueve
con `.to("cuda")` y cada lote se copia a la GPU dentro del ciclo.

In [5]:
import benchmark                                   # scripts/benchmark.py (train_epoch, evaluate, sync)
from asl.torch_models import MODELS, KERAS_PARAMS, count_params

print(inspect.getsource(benchmark.train_epoch))

def train_epoch(model, opt, loss_fn, X, y, device, batch, gen):
    """
    Una época. Devuelve (total, gather CPU, transferencia H2D, cómputo de la red), en segundos.

    - gather: armar el lote en la CPU (indexar el tensor).
    - h2d: copia CPU->GPU, con sync antes y después para aislarla.
    - step: forward + backward + Adam. En GPU se mide con eventos CUDA (tiempo real de los
      kernels); en CPU con el reloj, porque ahí no hay trabajo asíncrono.
    """
    model.train()
    perm = torch.randperm(len(X), generator=gen)
    t_gather = t_h2d = t_step_cpu = 0.0
    events = []
    cuda = device.type == "cuda"
    sync(device)
    t0 = time.perf_counter()
    for i in range(0, len(X), batch):
        idx = perm[i:i + batch]
        tg = time.perf_counter()
        xb_cpu, yb_cpu = X[idx], y[idx]
        t_gather += time.perf_counter() - tg

        sync(device)
        th = time.perf_counter()
        xb, yb = xb_cpu.to(device), yb_cpu.to(device)
        sync(device)
        t_h2

In [6]:
def run_torch(device_name, run):
    device = torch.device(device_name)
    cuda = device.type == "cuda"
    if cuda:
        torch.backends.cudnn.benchmark = True       # configuración CUDA
    res = {"device": device_name, "run": str(run), "framework": "torch", "torch": torch.__version__,
           "cpu_threads": torch.get_num_threads(), "gpu": GPU_NAME if cuda else None,
           "batch": BATCH, **DATA_PREP, "modelos": {}}

    for nombre, build in MODELS.items():
        torch.manual_seed(RANDOM_SEED)
        gen = torch.Generator().manual_seed(RANDOM_SEED)
        model = build().to(device)                  # los pesos pasan a la GPU solo aquí
        n_par = count_params(model)
        assert n_par == KERAS_PARAMS[nombre], (nombre, n_par)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        loss_fn = nn.CrossEntropyLoss()

        te, tg, th, ts = [], [], [], []
        t0 = time.perf_counter()
        for ep in range(EPOCHS):
            a, b, c, d = benchmark.train_epoch(model, opt, loss_fn, X_tr, y_tr, device, BATCH, gen)
            te.append(a); tg.append(b); th.append(c); ts.append(d)
            print(f"  {nombre} ép {ep + 1}: {a:.2f} s (gather {b:.2f}, H2D {c:.2f}, red {d:.2f})")
        t_train = time.perf_counter() - t0

        benchmark.sync(device)
        t1 = time.perf_counter()
        acc = benchmark.evaluate(model, X_te, y_te, device, 256)
        benchmark.sync(device)
        res["modelos"][nombre] = {
            "parametros": n_par, "epocas": EPOCHS, "train_seconds": round(t_train, 3),
            "epoch_seconds": [round(t, 3) for t in te], "gather_seconds": [round(t, 3) for t in tg],
            "h2d_seconds": [round(t, 3) for t in th], "step_seconds": [round(t, 3) for t in ts],
            "eval_seconds": round(time.perf_counter() - t1, 3), "accuracy": acc}
        print(f"{nombre}: {t_train:.1f} s, accuracy {acc * 100:.2f} %")
        del model, opt
        if cuda:
            torch.cuda.empty_cache()

    out = RESULTS / f"torch_{device_name}_run{run}.json"
    out.write_text(json.dumps(res, indent=2, ensure_ascii=False))
    print("Guardado en", out)

In [7]:
if RUN_TORCH_CPU:                                   # control: mismo código sin GPU
    for run in range(1, RUNS + 1):
        print(f"\n######## PyTorch CPU, corrida {run}/{RUNS} ########")
        run_torch("cpu", run)

In [8]:
if RUN_TORCH_CUDA:                                  # solución acelerada
    for run in range(1, RUNS + 1):
        print(f"\n######## PyTorch CUDA ({GPU_NAME}), corrida {run}/{RUNS} ########")
        run_torch("cuda", run)


######## PyTorch CUDA (NVIDIA GeForce RTX 4070 SUPER), corrida 1/3 ########


  CNN-1 Baseline ép 1: 0.81 s (gather 0.04, H2D 0.05, red 0.75)


  CNN-1 Baseline ép 2: 0.43 s (gather 0.04, H2D 0.05, red 0.37)


  CNN-1 Baseline ép 3: 0.43 s (gather 0.03, H2D 0.05, red 0.37)


  CNN-1 Baseline ép 4: 0.43 s (gather 0.03, H2D 0.06, red 0.37)


  CNN-1 Baseline ép 5: 0.43 s (gather 0.02, H2D 0.05, red 0.37)
CNN-1 Baseline: 2.5 s, accuracy 78.48 %


... (37 outputs left out)

  CNN-3 Regularizada ép 2: 1.48 s (gather 0.06, H2D 0.04, red 1.44)


  CNN-3 Regularizada ép 3: 1.49 s (gather 0.06, H2D 0.04, red 1.44)


  CNN-3 Regularizada ép 4: 1.49 s (gather 0.06, H2D 0.04, red 1.44)


  CNN-3 Regularizada ép 5: 1.49 s (gather 0.06, H2D 0.05, red 1.44)
CNN-3 Regularizada: 7.4 s, accuracy 56.41 %
Guardado en /home/Tono/Downloads/asl-cuda/results/torch_cuda_run3.json


## 4. CNN desde cero: NumPy (CPU) contra CuPy (GPU)

Vive en su propio notebook, [`03_cnn_scratch.ipynb`](03_cnn_scratch.ipynb): mismo algoritmo (`im2col` +
multiplicación de matrices, backward a mano, sin autograd) en NumPy, en CuPy y en CuPy con kernels CUDA propios.
Aquí solo se leen sus resultados (`scratch_*_run*.json`) para la tabla comparativa de la sección 7.

In [9]:
# Los benchmarks de NumPy y CuPy se corren en notebooks/03_cnn_scratch.ipynb
print(sorted(p.name for p in RESULTS.glob("scratch_*_run*.json")))

['scratch_cupy_kernels_run1.json', 'scratch_cupy_kernels_run2.json', 'scratch_cupy_kernels_run3.json', 'scratch_cupy_run1.json', 'scratch_cupy_run2.json', 'scratch_cupy_run3.json', 'scratch_numpy_run1.json', 'scratch_numpy_run2.json', 'scratch_numpy_run3.json']


## 5. Evidencia de uso de hardware

`scripts/evidence.py` muestrea cada 0.5 s los núcleos de CPU (psutil) y la GPU (`nvidia-smi`) y guarda
`results/evidence/usage.csv` y `usage.png`. Tiene dos fases: **data** (CPU multinúcleo saturada,
GPU ociosa) y **train** (GPU ocupada, la CPU sigue ocupada por el hilo de Python y OpenMP de PyTorch). El archivo `results/evidence/phase.txt` indica la
fase en curso.

**Capturas manuales (parte de la nota):** mientras corre esta celda, tomar capturas de pantalla del
**Administrador de Tareas** (pestaña Rendimiento, CPU con vista de procesadores lógicos) en la
fase `data`, y de `nvidia-smi` (`nvidia-smi -l 1` en otra terminal) en la fase `train`. Guardarlas en
`results/evidence/` como `captura_cpu_data.png`, `captura_gpu_train.png`, etc.

In [10]:
if RUN_EVIDENCE:
    sh([sys.executable, "scripts/evidence.py", "--epochs", 40])
    shutil.copy(EVIDENCE / "usage.png", EVIDENCE / "usage_ultima_corrida.png")

[data] Train 11,600 | Val 1,450 | Test 1,450
[fase] data


[fase] train


época 1/40: 1.86 s


época 2/40: 1.51 s


época 3/40: 1.51 s


época 4/40: 1.51 s


... (33 outputs left out)

época 38/40: 1.51 s


época 39/40: 1.51 s


época 40/40: 1.51 s
[fase] idle


Guardado en /home/Tono/Downloads/asl-cuda/results/evidence


## 6. Entrenamiento completo (Early Stopping y ReduceLROnPlateau)

Confirma que la aceleración no cambia la calidad del modelo. Deja `results/full_training_gpu.json`.

In [11]:
if RUN_FULL_TRAINING:
    sh([sys.executable, "scripts/train_full.py"])

## 7. Tablas de tiempos y speedup por fase

Speedup = `T_secuencial / T_paralelo`. Segundos por época sin la época 1, media y desviación entre corridas.

In [12]:
def cargar(patron):
    return [json.load(open(f)) for f in sorted(glob.glob(str(RESULTS / patron)))]

def por_epoca(runs, modelo, clave="epoch_seconds"):
    # media por corrida (sin la época 1) -> media y desviación entre corridas
    v = [np.mean(r["modelos"][modelo][clave][1:]) for r in runs if clave in r["modelos"].get(modelo, {})]
    return (np.mean(v), np.std(v, ddof=1) if len(v) > 1 else 0.0) if v else (np.nan, np.nan)

cfgs = {"Keras CPU (secuencial)": cargar("keras_cpu_run*.json"),
        "PyTorch CPU": cargar("torch_cpu_run*.json"),
        "PyTorch CUDA": cargar("torch_cuda_run*.json"),
        "NumPy CPU (desde cero)": cargar("scratch_numpy_run*.json"),
        "CuPy GPU (desde cero)": cargar("scratch_cupy_run*.json"),
        "CuPy GPU + kernels propios": cargar("scratch_cupy_kernels_run*.json")}

modelos = list(MODELS)
filas = []
for nombre, runs in cfgs.items():
    fila = {"Configuración": nombre, "Corridas": len(runs)}
    for m in modelos:
        mu, sd = por_epoca(runs, m)
        fila[m] = f"{mu:.3f} ± {sd:.3f}" if not np.isnan(mu) else "-"
    filas.append(fila)
print("Tiempo por época (s)")
display(pd.DataFrame(filas))

def speedup(num, den):
    out = {}
    for m in modelos:
        a, b = por_epoca(cfgs[num], m)[0], por_epoca(cfgs[den], m)[0]
        out[m] = f"{a / b:.2f}x" if not (np.isnan(a) or np.isnan(b)) else "-"
    return out

print("\nSpeedup del entrenamiento por época")
display(pd.DataFrame({
    "PyTorch CUDA vs Keras CPU (línea base)": speedup("Keras CPU (secuencial)", "PyTorch CUDA"),
    "PyTorch CUDA vs PyTorch CPU (mismo framework)": speedup("PyTorch CPU", "PyTorch CUDA"),
    "CuPy GPU vs NumPy CPU (desde cero)": speedup("NumPy CPU (desde cero)", "CuPy GPU (desde cero)"),
    "CuPy + kernels propios vs CuPy": speedup("CuPy GPU (desde cero)", "CuPy GPU + kernels propios"),
}))

Tiempo por época (s)


,Configuración,Corridas,CNN-1 Baseline,CNN-2 Profunda,CNN-3 Regularizada
0,Keras CPU (secuencial),3,4.085 ± 0.031,21.628 ± 0.211,22.496 ± 0.380
1,PyTorch CPU,1,9.583 ± 0.000,36.168 ± 0.000,34.106 ± 0.000
2,PyTorch CUDA,3,0.426 ± 0.003,1.464 ± 0.003,1.496 ± 0.017
3,NumPy CPU (desde cero),3,64.123 ± 5.070,-,-
4,CuPy GPU (desde cero),3,1.088 ± 0.024,4.755 ± 0.216,4.924 ± 0.027
5,CuPy GPU + kernels propios,3,0.691 ± 0.020,4.230 ± 0.121,4.422 ± 0.084



Speedup del entrenamiento por época


,PyTorch CUDA vs Keras CPU (línea base),PyTorch CUDA vs PyTorch CPU (mismo framework),CuPy GPU vs NumPy CPU (desde cero),CuPy + kernels propios vs CuPy
CNN-1 Baseline,9.58x,22.48x,58.94x,1.58x
CNN-2 Profunda,14.77x,24.70x,-,1.12x
CNN-3 Regularizada,15.03x,22.79x,-,1.11x


In [13]:
# Speedup por fase. Fase de datos: 1 worker (secuencial) contra N workers (CPU multinúcleo).
# Fase de entrenamiento: solo el cómputo de la red (step) en CPU contra GPU. La transferencia H2D
# solo existe en GPU y es un costo añadido por la paralelización, se reporta aparte.
t1 = sweep.segundos.iloc[0]
tN = sweep.segundos.iloc[-1]
print(f"Fase A, datos: {t1:.2f} s (1 worker) -> {tN:.2f} s ({WORKERS} workers) | speedup {t1 / tN:.2f}x")

filas = []
for m in modelos:
    cpu, gpu = cfgs["PyTorch CPU"], cfgs["PyTorch CUDA"]
    s_cpu, s_gpu = por_epoca(cpu, m, "step_seconds")[0], por_epoca(gpu, m, "step_seconds")[0]
    h2d = por_epoca(gpu, m, "h2d_seconds")[0]
    g = por_epoca(gpu, m, "gather_seconds")[0]
    tot = por_epoca(gpu, m)[0]
    filas.append({"Modelo": m, "step CPU (s)": round(s_cpu, 3), "step GPU (s)": round(s_gpu, 3),
                  "Speedup step": f"{s_cpu / s_gpu:.1f}x", "gather (s)": round(g, 3),
                  "H2D (s)": round(h2d, 3), "% H2D": f"{100 * h2d / tot:.1f}%", "época GPU (s)": round(tot, 3)})
print("\nFase B, entrenamiento por época")
display(pd.DataFrame(filas))

Fase A, datos: 3.32 s (1 worker) -> 0.53 s (20 workers) | speedup 6.32x

Fase B, entrenamiento por época


,Modelo,step CPU (s),step GPU (s),Speedup step,gather (s),H2D (s),% H2D,época GPU (s)
0,CNN-1 Baseline,9.552,0.369,25.9x,0.030,0.054,12.6%,0.426
1,CNN-2 Profunda,36.142,1.415,25.5x,0.055,0.045,3.1%,1.464
2,CNN-3 Regularizada,34.083,1.447,23.6x,0.055,0.045,3.0%,1.496


In [14]:
# Regenera summary.json y las gráficas del informe con los datos de esta máquina
sh([sys.executable, "scripts/aggregate.py"])
sh([sys.executable, "scripts/generate_report_charts.py"])
print("Listo. Gráficas en results/charts/ y resumen en results/summary.json")

Config                Modelo                         s/época  H2D s/ép  Speedup     Acc
-----------------------------------------------------------------------------
keras_cpu             CNN-1 Baseline             4.08 ± 0.06          -     1.0x   85.9%
keras_cpu             CNN-2 Profunda            21.63 ± 0.27          -     1.0x   50.7%
keras_cpu             CNN-3 Regularizada        22.50 ± 0.34          -     1.0x   57.7%
                      (3 corridas; data_prep_seconds=0.44s)

scratch_cupy_kernels  CNN-1 Baseline             0.69 ± 0.03      0.069     5.9x   81.4%
scratch_cupy_kernels  CNN-2 Profunda             4.23 ± 0.17      0.073     5.1x   22.5%
scratch_cupy_kernels  CNN-3 Regularizada         4.42 ± 0.13      0.076     5.1x   36.5%
                      (3 corridas; )

scratch_cupy          CNN-1 Baseline             1.09 ± 0.03      0.066     3.8x   84.1%
scratch_cupy          CNN-2 Profunda             4.76 ± 0.19      0.071     4.5x   66.1%
scratch_cupy          C

Guardado: /home/Tono/Downloads/asl-cuda/results/charts/epoch_times_comparison.png
Guardado: /home/Tono/Downloads/asl-cuda/results/charts/speedup_vs_baseline.png
Guardado: /home/Tono/Downloads/asl-cuda/results/charts/phase_breakdown_gpu.png
Guardado: /home/Tono/Downloads/asl-cuda/results/charts/training_convergence.png
Todos los gráficos han sido generados exitosamente en results/charts/
Listo. Gráficas en results/charts/ y resumen en results/summary.json


### Después de correrlo

1. Revisar que `GPU_NAME` (sección 0) sea la que debe aparecer en el informe.
2. Guardar las capturas de pantalla en `results/evidence/`.
3. Hacer `git add results/ notebooks/` (con las salidas del notebook guardadas) y avisar para actualizar el informe.